# Model Training - MHGSL

This notebook trains the MHGSL model for JKN fraud detection.

In [ ]:
import sys
sys.path.append('../backend/src')

import torch
import numpy as np
from data.pipeline import MHGSLDataPipeline
from models.mhgsl import MHGSLModel, MHGSLTrainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Fetch and prepare data
pipeline = MHGSLDataPipeline()
data = pipeline.run_pipeline()

features_df = data['features']
print(f"Features shape: {features_df.shape}")

In [ ]:
# Prepare data for training
# (This is a simplified example - actual implementation depends on data structure)
num_nodes = len(features_df)
input_dim = 16  # Number of features

# Generate synthetic features for demonstration
X = torch.randn(num_nodes, input_dim)
y = torch.randint(0, 2, (num_nodes,))  # Binary classification

# Generate synthetic edge indices
edge_index = torch.randint(0, num_nodes, (2, num_nodes * 2))

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"edge_index shape: {edge_index.shape}")

In [ ]:
# Split data
train_idx, test_idx = train_test_split(range(num_nodes), test_size=0.2, random_state=42)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True

print(f"Training samples: {train_mask.sum()}")
print(f"Test samples: {test_mask.sum()}")

In [ ]:
# Initialize model
model = MHGSLModel(
    input_dim=input_dim,
    hidden_dim=128,
    num_layers=3,
    num_heads=4,
    num_classes=2
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize trainer
trainer = MHGSLTrainer(model, learning_rate=0.001)

In [ ]:
# Training loop
epochs = 100
train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(epochs):
    # Train
    train_loss = trainer.train_epoch(X, (edge_index, edge_index, edge_index), y)
    train_losses.append(train_loss)
    
    # Evaluate
    test_loss, test_acc = trainer.evaluate(X, (edge_index, edge_index, edge_index), y)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Test Loss = {test_loss:.4f}, Test Acc = {test_acc:.4f}")

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()

ax2.plot(test_accuracies, label='Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
torch.save(model.state_dict(), '../backend/models/mhgsl_model.pth')
print("Model saved!")